# NYC Taxi Databricks Analytics

**Purpose:** build a Databricks lakehouse workflow for NYC green and yellow taxi trip data, from raw Parquet ingestion through Delta tables, borough-level analytics, and scalable Spark-based modeling.

This notebook is the **executable source of truth**. It keeps configuration near the top, writes reusable Delta artifacts to Unity Catalog volumes, and keeps interpretation in Markdown cells so the code path remains reviewable.


## 1. Lakehouse Build

This section creates the **governed Databricks storage layout**, loads raw taxi files, standardizes green/yellow schemas, and applies quality rules before writing the curated Delta table.


### 1.1 Setup

Configure **imports**, Spark runtime behavior, Unity Catalog paths, source files, and execution flags before running ingestion or modeling cells.


In [ ]:
import datetime
import importlib.util
import json
import subprocess
import sys
from typing import Optional

import numpy as np
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import broadcast

# Align temporal features with NYC local time.
spark.conf.set("spark.sql.session.timeZone", "America/New_York")


In [ ]:
# Update these values if your Unity Catalog layout differs.
CATALOG = "workspace"
SCHEMA = "bde"
VOLUME = "nyc_taxi"

# Configure execution modes.
RUN_DOWNLOADS = True
ALLOW_RUNTIME_PIP_INSTALL = True
OVERWRITE_TABLES = True
RUN_PROFILE_PREVIEWS = False
RUN_MODEL_DIAGNOSTICS = False

RAW_GREEN_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/green"
RAW_YELLOW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/yellow"
TAXI_ZONE_LOOKUP_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/taxi_zone_lookup.csv"
)

TAXI_ZONE_TABLE = f"{CATALOG}.{SCHEMA}.taxi_zone_lookup"
TRIPS_TABLE = f"{CATALOG}.{SCHEMA}.taxi_trips_cleaned_borough"
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/models/model_a_ridge_v1"

# Raw source files are downloaded only when RUN_DOWNLOADS is enabled.
GREEN_TAXI_FILES = [
    {
        "file_id": "1T7o40ZqmAK90W5o9TBOPrlpECBZ9KedV",
        "filename": "green_taxi.parquet",
    }
]
YELLOW_TAXI_FILES = [
    {
        "file_id": "1Iayo8ZaL4oOB2v65VpnumdDSOVz3xoNf",
        "filename": "yellow_taxi.parquet",
    }
]

write_mode = "overwrite" if OVERWRITE_TABLES else "errorifexists"

# Ensure required lakehouse objects exist before ingestion.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

dbutils.fs.mkdirs(RAW_GREEN_PATH)
dbutils.fs.mkdirs(RAW_YELLOW_PATH)


In [ ]:
# Install and define download helpers when source ingestion is enabled.
if RUN_DOWNLOADS:
    if importlib.util.find_spec("gdown") is None:
        if not ALLOW_RUNTIME_PIP_INSTALL:
            raise ImportError(
                "gdown is required. Enable ALLOW_RUNTIME_PIP_INSTALL or install it "
                "on the cluster before running ingestion."
            )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown"])

    from gdown import download
else:
    download = None


def path_exists(path: str) -> bool:
    """Return whether a Databricks filesystem path exists."""
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def fetch_to_volume(file_id: str, destination_dir: str, filename: str) -> str:
    """Download a Google Drive file into a Databricks volume path.

    Args:
        file_id: Google Drive file identifier.
        destination_dir: Databricks volume directory.
        filename: Output filename inside the destination directory.

    Returns:
        Fully qualified Databricks volume path for the downloaded file.
    """
    destination_path = f"{destination_dir}/{filename}"
    if path_exists(destination_path):
        print(f"Source file already exists, skipping download: {destination_path}")
        return destination_path

    if download is None:
        raise RuntimeError("RUN_DOWNLOADS must be True to fetch missing source files.")

    url = f"https://drive.google.com/uc?id={file_id}"
    download(url, destination_path, quiet=False)
    return destination_path


def fetch_sources(sources: list[dict[str, str]], destination_dir: str) -> None:
    """Download configured source files when ingestion downloads are enabled."""
    if not RUN_DOWNLOADS:
        print(f"Skipping downloads for {destination_dir}; RUN_DOWNLOADS is False.")
        return

    for source in sources:
        fetch_to_volume(source["file_id"], destination_dir, source["filename"])


In [ ]:
# Download or reuse configured green taxi source files.
fetch_sources(GREEN_TAXI_FILES, RAW_GREEN_PATH)


In [ ]:
# Download or reuse configured yellow taxi source files.
fetch_sources(YELLOW_TAXI_FILES, RAW_YELLOW_PATH)


In [ ]:
# Confirm raw source files are present in the configured volume paths.
print("Raw green taxi files")
display(dbutils.fs.ls(RAW_GREEN_PATH))

print("Raw yellow taxi files")
display(dbutils.fs.ls(RAW_YELLOW_PATH))


In [ ]:
# Load raw source files as Spark DataFrames.
green_df = spark.read.parquet(RAW_GREEN_PATH)
yellow_df = spark.read.parquet(RAW_YELLOW_PATH)

print("Raw taxi Parquet files loaded into Spark DataFrames.")


In [ ]:
# Display raw row samples when profile previews are enabled.
if RUN_PROFILE_PREVIEWS:
    display(green_df.limit(5))
    display(yellow_df.limit(5))


In [ ]:
# Inspect raw schemas and sample rows when profile previews are enabled.
if RUN_PROFILE_PREVIEWS:
    print("Green taxi schema")
    green_df.printSchema()
    display(green_df.limit(5))

    print("Yellow taxi schema")
    yellow_df.printSchema()
    display(yellow_df.limit(5))


### 1.2 Taxi Zone Lookup to Delta

Load the NYC taxi zone reference file once, validate that each `LocationID` is unique, and persist it as a **Delta table** for repeatable borough enrichment.


In [ ]:
# Load and validate the taxi zone lookup reference data.
taxi_zone_schema = T.StructType([
    T.StructField("LocationID", T.IntegerType(), False),
    T.StructField("Borough", T.StringType(), True),
    T.StructField("Zone", T.StringType(), True),
    T.StructField("service_zone", T.StringType(), True),
])

taxi_zone_df = (
    spark.read
    .schema(taxi_zone_schema)
    .option("header", True)
    .csv(TAXI_ZONE_LOOKUP_PATH)
)

location_count = taxi_zone_df.count()
distinct_location_count = taxi_zone_df.select("LocationID").distinct().count()
assert distinct_location_count == location_count, (
    "Duplicate LocationID values found in taxi_zone_lookup"
)

taxi_zone_df.write.mode(write_mode).saveAsTable(TAXI_ZONE_TABLE)
print(f"Taxi zones loaded: {spark.table(TAXI_ZONE_TABLE).count():,}")


In [ ]:
%sql
-- Inspect taxi zone lookup rows for handover evidence.
SELECT *
FROM taxi_zone_lookup
ORDER BY LocationID;


### 1.3 Schema Harmonization and Union

Green and yellow taxi files use different source column names. This step maps both services into one **shared trip schema** so downstream SQL and modeling can operate on a single table.


In [ ]:
# Map yellow taxi columns into the shared trip schema.
yellow_std = (
    yellow_df
    .withColumn("color", F.lit("yellow"))
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
)

# Map green taxi columns into the shared trip schema.
green_std = (
    green_df
    .withColumn("color", F.lit("green"))
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("trip_type", F.col("trip_type").cast("int"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
)

trips_raw = yellow_std.unionByName(green_std, allowMissingColumns=True)
trips_raw.createOrReplaceTempView("trips_raw")


In [ ]:
# Count raw trips by taxi color as a scale validation checkpoint.
raw_color_counts = trips_raw.groupBy("color").count().orderBy("color")
display(raw_color_counts)

if RUN_PROFILE_PREVIEWS:
    trips_raw.printSchema()
    trips_raw.show(5, truncate=False)


### 1.4 Trip Feature Engineering

Derive operational features such as **duration**, **distance in kilometers**, **average speed**, calendar fields, and fare efficiency. These fields support both business analysis and machine learning.


In [ ]:
%sql
-- Engineer normalized timestamps, duration, distance, and speed.
CREATE OR REPLACE TEMP VIEW trips_enriched AS
SELECT
  *,
  CAST(pickup_datetime AS TIMESTAMP) AS pickup_ts,
  CAST(dropoff_datetime AS TIMESTAMP) AS dropoff_ts,
  (CAST(dropoff_datetime AS BIGINT) - CAST(pickup_datetime AS BIGINT)) / 60.0 AS duration_min,
  trip_distance * 1.60934 AS trip_distance_km,
  try_divide(
    trip_distance * 1.60934,
    NULLIF((CAST(dropoff_datetime AS BIGINT) - CAST(pickup_datetime AS BIGINT)) / 3600.0, 0)
  ) AS speed_kmh
FROM trips_raw;


### 1.5 Quality Filters

The quality filter removes unrealistic trips without dropping records only because unused fields are missing. The filter logic is intentionally conservative and reports row retention explicitly:

- keep only trips where drop-off occurs after pickup;
- keep realistic duration values from **1 to 180 minutes**;
- keep realistic distance values from **0.1 to 200 km**;
- remove negative speeds and cap speed at **120 km/h**, which is above normal NYC limits but allows airport/highway trips and avoids over-filtering;
- allow nullable passenger counts, but constrain populated values to **0-6**;
- require non-negative `total_amount` because the ML target is fare total;
- keep valid TLC taxi zone IDs when location IDs are present;
- enforce service date windows: yellow from **2009 onward**, green from **August 2013 onward**, both before 2025;
- validate rate code, payment type, trip type, and store-and-forward flags where populated.

The next audit cell reports raw rows, clean rows, removed rows, and removed percentage so the cleaning impact is explicit.


In [ ]:
%sql
-- Inspect raw temporal bounds before quality filtering.
WITH b AS (
  SELECT MIN(pickup_ts) AS min_pickup, MAX(dropoff_ts) AS max_dropoff
  FROM trips_enriched
)
SELECT * FROM b;


In [ ]:
%sql
-- Apply conservative quality filters while preserving nullable business fields.
CREATE OR REPLACE TEMP VIEW trips_clean AS
SELECT *
FROM trips_enriched
WHERE
  dropoff_ts > pickup_ts
  AND duration_min BETWEEN 1 AND 180
  AND trip_distance_km BETWEEN 0.1 AND 200
  AND speed_kmh > 0
  AND speed_kmh <= 120
  AND (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 6)
  AND total_amount >= 0
  AND (PULocationID BETWEEN 1 AND 265 OR PULocationID IS NULL)
  AND (DOLocationID BETWEEN 1 AND 265 OR DOLocationID IS NULL)
  AND (
    (
      color = 'yellow'
      AND pickup_ts >= TIMESTAMP('2009-01-01')
      AND dropoff_ts < TIMESTAMP('2025-01-01')
    )
    OR (
      color = 'green'
      AND pickup_ts >= TIMESTAMP('2013-08-01')
      AND dropoff_ts < TIMESTAMP('2025-01-01')
    )
  )
  AND (RatecodeID IS NULL OR CAST(RatecodeID AS INT) IN (1, 2, 3, 4, 5, 6, 99))
  AND (payment_type IS NULL OR CAST(payment_type AS INT) IN (0, 1, 2, 3, 4, 5, 6))
  AND (trip_type IS NULL OR CAST(trip_type AS INT) IN (1, 2))
  AND (store_and_fwd_flag IS NULL OR store_and_fwd_flag IN ('Y', 'N'));


In [ ]:
%sql
-- Summarize row retention after quality filters.
WITH base AS (
  SELECT COUNT(*) AS n FROM trips_enriched
),
clean AS (
  SELECT COUNT(*) AS n FROM trips_clean
)
SELECT
  base.n AS raw_rows,
  clean.n AS clean_rows,
  (base.n - clean.n) AS removed_rows,
  ROUND((base.n - clean.n) / base.n * 100, 2) AS removed_pct
FROM base, clean;


### 1.6 Borough Table

Join pickup and drop-off locations to the taxi zone lookup, fill missing borough metadata with `Unknown`, and save the cleaned borough-enriched table for **analytics and modeling**.


In [ ]:
# Enrich trips with borough metadata and persist the curated Delta table.
spark.sql("""
CREATE OR REPLACE TEMP VIEW trips_enriched_borough AS
SELECT
  t.*,
  COALESCE(pu.Borough, 'Unknown') AS pu_borough,
  COALESCE(do.Borough, 'Unknown') AS do_borough,
  COALESCE(pu.Zone, 'Unknown') AS pu_zone,
  COALESCE(do.Zone, 'Unknown') AS do_zone
FROM trips_clean t
LEFT JOIN taxi_zone_lookup pu
  ON t.PULocationID = pu.LocationID
LEFT JOIN taxi_zone_lookup do
  ON t.DOLocationID = do.LocationID
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW trips_final AS
SELECT
  e.*,
  year(pickup_ts) AS year,
  month(pickup_ts) AS month
FROM trips_enriched_borough e
""")

if OVERWRITE_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {TRIPS_TABLE}")

spark.sql(f"""
CREATE TABLE {TRIPS_TABLE}
USING DELTA
PARTITIONED BY (year, month)
AS SELECT * FROM trips_final
""")

display(spark.sql(f"SELECT COUNT(*) AS final_rows FROM {TRIPS_TABLE}"))


## 2. Business Analytics

This section answers reviewer-facing questions about **demand**, **revenue**, **tips**, route flows, and operational efficiency using the curated Delta table.


### 2.1 Monthly Trip and Revenue Summary

Summarize trip volume, revenue, passenger patterns, and peak operating periods by month to show seasonality and scale.


In [ ]:
%sql
-- Produce the required monthly operating summary.
WITH base AS (
  SELECT
    year,
    month,
    make_date(year, month, 1) AS ym,
    date_format(pickup_ts, 'EEEE') AS dow_name,
    hour(pickup_ts) AS hr,
    passenger_count,
    total_amount
  FROM taxi_trips_cleaned_borough
),
agg AS (
  SELECT
    ym,
    COUNT(*) AS total_trips,
    AVG(passenger_count) AS avg_passengers,
    AVG(total_amount) AS avg_amount_per_trip,
    AVG(CASE WHEN passenger_count > 0 THEN try_divide(total_amount, passenger_count) END) AS avg_amount_per_passenger
  FROM base
  GROUP BY ym
),
dow_rank AS (
  SELECT
    ym,
    dow_name,
    COUNT(*) AS trips_by_dow,
    ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, dow_name) AS rn
  FROM base
  GROUP BY ym, dow_name
),
hr_rank AS (
  SELECT
    ym,
    hr,
    COUNT(*) AS trips_by_hr,
    ROW_NUMBER() OVER (PARTITION BY ym ORDER BY COUNT(*) DESC, hr) AS rn
  FROM base
  GROUP BY ym, hr
)
SELECT
  a.ym,
  a.total_trips,
  d.dow_name AS most_trips_dow,
  h.hr AS most_trips_hour,
  ROUND(a.avg_passengers, 2) AS avg_passengers,
  ROUND(a.avg_amount_per_trip, 2) AS avg_amount_per_trip,
  ROUND(a.avg_amount_per_passenger, 2) AS avg_amount_per_passenger
FROM agg a
LEFT JOIN dow_rank d ON a.ym = d.ym AND d.rn = 1
LEFT JOIN hr_rank h ON a.ym = h.ym AND h.rn = 1
ORDER BY a.ym;


### 2.2 Descriptive Statistics by Taxi Color

Compare green and yellow taxi services across duration, distance, speed, and fare distributions.


In [ ]:
%sql
-- Produce required taxi-color distribution statistics.
WITH base AS (
  SELECT color, duration_min, trip_distance_km, speed_kmh
  FROM taxi_trips_cleaned_borough
  WHERE duration_min IS NOT NULL
    AND trip_distance_km IS NOT NULL
    AND speed_kmh IS NOT NULL
)
SELECT
  color,
  ROUND(AVG(duration_min), 2) AS duration_avg_min,
  ROUND(percentile_approx(duration_min, 0.5, 10000), 2) AS duration_med_min,
  ROUND(MIN(duration_min), 2) AS duration_min_min,
  ROUND(MAX(duration_min), 2) AS duration_max_min,
  ROUND(AVG(trip_distance_km), 2) AS dist_avg_km,
  ROUND(percentile_approx(trip_distance_km, 0.5, 10000), 2) AS dist_med_km,
  ROUND(MIN(trip_distance_km), 2) AS dist_min_km,
  ROUND(MAX(trip_distance_km), 2) AS dist_max_km,
  ROUND(AVG(speed_kmh), 2) AS speed_avg_kmh,
  ROUND(percentile_approx(speed_kmh, 0.5, 10000), 2) AS speed_med_kmh,
  ROUND(MIN(speed_kmh), 2) AS speed_min_kmh,
  ROUND(MAX(speed_kmh), 2) AS speed_max_kmh
FROM base
GROUP BY color
ORDER BY color;


### 2.3 Route Demand Grid

Create a granular route-time demand grid that can support operational planning, dashboarding, and baseline model features.


In [ ]:
%sql
-- Produce the required route-time demand and revenue grid.
SELECT
  color,
  pu_borough,
  do_borough,
  make_date(year, month, 1) AS ym,
  date_format(pickup_ts, 'EEEE') AS dow_name,
  hour(pickup_ts) AS hr,
  COUNT(*) AS total_trips,
  ROUND(AVG(trip_distance_km), 2) AS avg_distance_km,
  ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,
  ROUND(SUM(total_amount), 2) AS total_amount
FROM taxi_trips_cleaned_borough
GROUP BY color, pu_borough, do_borough, year, month, date_format(pickup_ts, 'EEEE'), hour(pickup_ts)
ORDER BY ym, color, pu_borough, do_borough, dow_name, hr;


### 2.4 Top 2024 Revenue Routes

Rank borough-pair routes by 2024 revenue contribution to identify the most commercially important flows.


In [ ]:
%sql
-- Rank 2024 borough-pair routes by revenue share.
WITH y2024 AS (
  SELECT *
  FROM taxi_trips_cleaned_borough
  WHERE year = 2024
),
pair_rev AS (
  SELECT pu_borough, do_borough, SUM(total_amount) AS pair_total
  FROM y2024
  GROUP BY pu_borough, do_borough
),
overall AS (
  SELECT SUM(pair_total) AS overall_total FROM pair_rev
),
top10 AS (
  SELECT
    pu_borough,
    do_borough,
    pair_total,
    ROW_NUMBER() OVER (ORDER BY pair_total DESC) AS rn
  FROM pair_rev
  ORDER BY pair_total DESC
  LIMIT 10
)
SELECT
  t.pu_borough,
  t.do_borough,
  ROUND(t.pair_total, 2) AS pair_total_amount_2024,
  ROUND(t.pair_total / o.overall_total * 100, 2) AS share_pct_2024
FROM top10 t
CROSS JOIN overall o
ORDER BY t.pair_total DESC;


### 2.5 Tip Percentage Analysis

Measure how often trips include tips and how frequently tipped trips exceed a meaningful percentage threshold.


In [ ]:
%sql
-- Measure tip participation and high-tip share.
WITH nums AS (
  SELECT
    COUNT(*) AS n_all,
    SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) AS n_tipped,
    SUM(CASE WHEN tip_amount >= 15 THEN 1 ELSE 0 END) AS n_tip_15_plus
  FROM taxi_trips_cleaned_borough
)
SELECT
  ROUND((n_tipped * 100.0) / NULLIF(n_all, 0), 2) AS pct_with_tips,
  ROUND((n_tip_15_plus * 100.0) / NULLIF(n_tipped, 0), 2) AS pct_tips_15_plus_among_tipped
FROM nums;


### 2.6 Speed and Fare Efficiency

Group trips by duration bands to compare average speed, average distance per dollar, and revenue per hour. The added revenue-per-hour field supports the driver recommendation question directly.


In [ ]:
%sql
-- Compare speed, distance efficiency, and revenue efficiency by duration bin.
WITH binned AS (
  SELECT
    CASE
      WHEN duration_min < 5 THEN 'Under 5 mins'
      WHEN duration_min >= 5 AND duration_min < 10 THEN '5-10 mins'
      WHEN duration_min >= 10 AND duration_min < 20 THEN '10-20 mins'
      WHEN duration_min >= 20 AND duration_min < 30 THEN '20-30 mins'
      WHEN duration_min >= 30 AND duration_min < 60 THEN '30-60 mins'
      ELSE '60+ mins'
    END AS duration_bin,
    speed_kmh,
    trip_distance_km,
    total_amount,
    duration_min
  FROM taxi_trips_cleaned_borough
),
metrics AS (
  SELECT
    duration_bin,
    ROUND(AVG(speed_kmh), 2) AS avg_speed_kmh,
    ROUND(AVG(CASE WHEN total_amount > 0 THEN trip_distance_km / total_amount END), 4) AS avg_km_per_dollar,
    ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,
    ROUND(AVG(CASE WHEN duration_min > 0 THEN total_amount / duration_min * 60 END), 2) AS avg_amount_per_hour
  FROM binned
  GROUP BY duration_bin
)
SELECT
  duration_bin,
  avg_speed_kmh,
  avg_km_per_dollar,
  avg_amount_per_trip,
  avg_amount_per_hour,
  ROW_NUMBER() OVER (ORDER BY avg_amount_per_hour DESC) AS income_rank
FROM metrics
ORDER BY
  CASE duration_bin
    WHEN 'Under 5 mins' THEN 1
    WHEN '5-10 mins' THEN 2
    WHEN '10-20 mins' THEN 3
    WHEN '20-30 mins' THEN 4
    WHEN '30-60 mins' THEN 5
    ELSE 6
  END;


## 3. Machine Learning

The modeling path predicts `total_amount` with **train-only preprocessing**, time-based validation, interpretable baselines, and lightweight Spark-friendly model implementations.


### 3.1 Reusable Helpers

Define shared modeling helpers once so all candidate models use the same **scoring** and **RMSE** logic. Imports are centralized in the setup section.


In [ ]:
# Shared modeling helpers.
TABLE = TRIPS_TABLE


def predict_linear(
    df: DataFrame,
    feature_cols: list[str],
    weights,
    output_col: Optional[str] = None,
    out_col: Optional[str] = None,
) -> DataFrame:
    """Score a Spark DataFrame with a linear model.

    Args:
        df: Input Spark DataFrame.
        feature_cols: Ordered feature columns used by the model.
        weights: Model coefficients in the same order as feature_cols.
        output_col: Name of the generated prediction column.
        out_col: Backward-compatible alias for output_col.

    Returns:
        DataFrame with the prediction column appended.
    """
    target_col = output_col or out_col
    if target_col is None:
        raise ValueError("Pass output_col or out_col when scoring a linear model.")

    weight_values = [float(value) for value in np.asarray(weights).tolist()]
    prediction = None
    for weight, feature_col in zip(weight_values, feature_cols):
        term = F.lit(weight) * F.col(feature_col)
        prediction = term if prediction is None else prediction + term
    return df.withColumn(target_col, prediction)


def rmse(df: DataFrame, prediction_col: str, label_col: str) -> float:
    """Compute root mean squared error for Spark columns."""
    squared_error = F.pow(F.col(prediction_col) - F.col(label_col), 2)
    return float(
        df.select(squared_error.alias("squared_error"))
        .agg(F.sqrt(F.avg("squared_error")).alias("rmse"))
        .first()["rmse"]
    )


### 3.2 Modeling Splits

Use **time-based splits** to mimic forward-looking evaluation: train on historical trips, validate on September 2024, and test on October-December 2024.


In [ ]:
# Build time-based splits and base model columns.
base = (
    spark.table(TABLE)
    .select(
        "total_amount",
        "trip_distance_km",
        "duration_min",
        F.coalesce(F.col("passenger_count"), F.lit(1)).alias("passenger_count"),
        "color",
        "pu_borough",
        "do_borough",
        "year",
        "month",
        F.dayofweek("pickup_ts").alias("dow"),
        F.hour("pickup_ts").alias("hour"),
        "pickup_ts",
    )
    .where(
        "total_amount is not null "
        "and duration_min is not null "
        "and trip_distance_km is not null"
    )
)

train = base.where("pickup_ts < timestamp('2024-09-01')")
val = base.where(
    "pickup_ts >= timestamp('2024-09-01') "
    "and pickup_ts < timestamp('2024-10-01')"
)
test = base.where(
    "pickup_ts >= timestamp('2024-10-01') "
    "and pickup_ts < timestamp('2025-01-01')"
)

split_counts = (
    train.select(F.lit("train").alias("split"))
    .unionByName(val.select(F.lit("validation").alias("split")))
    .unionByName(test.select(F.lit("test").alias("split")))
    .groupBy("split")
    .count()
    .orderBy("split")
)
display(split_counts)


### 3.3 Baseline Model

Estimate average fare by route, taxi color, month, weekday, and hour. The baseline backs off to coarser groupings and finally the global mean so every row receives a **prediction**.


In [ ]:
# Use increasingly general route-time averages as fallback predictions.
KEY_LEVELS = [
    ["color_k", "pu_k", "do_k", "month", "dow", "hour"],
    ["color_k", "pu_k", "do_k", "month", "dow"],
    ["color_k", "pu_k", "do_k", "month"],
    ["color_k", "month"],
]


def add_key_norm_cols(df: DataFrame) -> DataFrame:
    """Normalize nullable categorical join keys used by baseline maps."""
    return (
        df.withColumn("color_k", F.coalesce(F.col("color"), F.lit("Unknown")))
        .withColumn("pu_k", F.coalesce(F.col("pu_borough"), F.lit("Unknown")))
        .withColumn("do_k", F.coalesce(F.col("do_borough"), F.lit("Unknown")))
    )


train_baseline = add_key_norm_cols(train)

baseline_maps = []
for keys in KEY_LEVELS:
    mapping = train_baseline.groupBy(*keys).agg(F.avg("total_amount").alias("pred"))
    baseline_maps.append((keys, broadcast(mapping)))

baseline_global_mean = float(train_baseline.agg(F.avg("total_amount")).first()[0])


def apply_baseline(df_in: DataFrame) -> DataFrame:
    """Apply hierarchical fallback averages from specific keys to global mean."""
    df = add_key_norm_cols(df_in)
    prediction_cols = []
    for i, (keys, mapping) in enumerate(baseline_maps):
        pred_col = f"pred_l{i}"
        df = df.join(mapping.withColumnRenamed("pred", pred_col), on=keys, how="left")
        prediction_cols.append(pred_col)

    return df.withColumn(
        "prediction_baseline",
        F.coalesce(
            *[F.col(col_name) for col_name in prediction_cols],
            F.lit(baseline_global_mean),
        ),
    )


val_b = apply_baseline(val)
test_b = apply_baseline(test)

rmse_val_baseline = rmse(
    val_b,
    prediction_col="prediction_baseline",
    label_col="total_amount",
)
rmse_test_baseline = rmse(
    test_b,
    prediction_col="prediction_baseline",
    label_col="total_amount",
)

print(
    f"Baseline RMSE - val: {rmse_val_baseline:.3f} | "
    f"test: {rmse_test_baseline:.3f}"
)


In [ ]:
# Inspect fare distributions before robust label clipping when diagnostics are enabled.
if RUN_MODEL_DIAGNOSTICS:
    for name, df in [("train", train), ("val", val), ("test", test)]:
        print(f"\n{name.upper()}")
        df.selectExpr(
            "AVG(total_amount) AS mean_amt",
            "STDDEV_POP(total_amount) AS std_amt",
            "percentile_approx(total_amount, 0.5, 1000) AS p50",
            "percentile_approx(total_amount, 0.9, 1000) AS p90",
            "percentile_approx(total_amount, 0.99, 1000) AS p99",
            "MAX(total_amount) AS max_amt",
        ).show(truncate=False)


In [ ]:
# Add robust labels by clipping extreme training fares.
cap_row = (
    train
    .selectExpr("percentile_approx(total_amount, 0.9995, 10000) AS p9995")
    .first()
)
CAP = float(cap_row["p9995"]) if cap_row and cap_row["p9995"] is not None else 500.0

print(f"Using label cap (train 99.95th): {CAP:.2f}")


def clip_amount(col, cap: float):
    """Clip values above the train-fitted robust label cap."""
    return F.when(col > F.lit(cap), F.lit(cap)).otherwise(col)


train = train.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
val = val.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
test = test.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))


### 3.4 Target Encoding

Fit smoothed target encodings on the **training set only**, then join those mappings into validation and test data with a global-mean fallback for unseen categories.


In [ ]:
# Fit smoothed target encodings on training data only.
TE_COLS = ["color", "pu_borough", "do_borough"]
y_global = float(train.agg(F.avg("label_robust")).first()[0])


def fit_te_smoothed(
    df_train: DataFrame,
    col_name: str,
    smoothing: float = 30.0,
) -> DataFrame:
    """Fit smoothed target-encoding values from training data only."""
    encoded = (
        df_train.groupBy(col_name)
        .agg(
            F.sum("label_robust").alias("sum_y"),
            F.count(F.lit(1)).alias("cnt"),
        )
    )
    return encoded.select(
        F.col(col_name),
        (
            (F.col("sum_y") + F.lit(smoothing) * F.lit(y_global))
            / (F.col("cnt") + F.lit(smoothing))
        ).alias(f"{col_name}_te"),
    )


te_smoothing = 30.0
te_maps = {
    col_name: fit_te_smoothed(train, col_name, smoothing=te_smoothing)
    for col_name in TE_COLS
}


def apply_target_encoding(df_in: DataFrame) -> DataFrame:
    """Join training-fitted target encodings with global fallback values."""
    df = df_in
    for col_name in TE_COLS:
        encoded_col = f"{col_name}_te"
        df = df.join(te_maps[col_name], on=col_name, how="left")
        df = df.withColumn(encoded_col, F.coalesce(F.col(encoded_col), F.lit(y_global)))
    return df


def rename_borough_te_columns(df: DataFrame) -> DataFrame:
    """Shorten target-encoded borough feature names."""
    return (
        df.withColumnRenamed("pu_borough_te", "pu_te")
        .withColumnRenamed("do_borough_te", "do_te")
    )


train_te = rename_borough_te_columns(apply_target_encoding(train))
val_te = rename_borough_te_columns(apply_target_encoding(val))
test_te = rename_borough_te_columns(apply_target_encoding(test))


### 3.5 Standardization

Compute means and standard deviations on **training data only**, then apply the same z-score transformation to validation and test splits.


In [ ]:
# Fit standardization statistics on training data only.
NUM_COLS = [
    "trip_distance_km",
    "duration_min",
    "passenger_count",
    "hour",
    "month",
    "dow",
    "color_te",
    "pu_te",
    "do_te",
]

stats = (
    train_te.agg(
        *[F.avg(col_name).alias(f"{col_name}_mean") for col_name in NUM_COLS],
        *[F.stddev_pop(col_name).alias(f"{col_name}_std") for col_name in NUM_COLS],
    )
    .collect()[0]
    .asDict()
)


def positive_stddev(col_name: str) -> float:
    """Return a non-zero training standard deviation for z-scoring."""
    std = stats[f"{col_name}_std"]
    return float(std) if std and std > 0 else 1.0


def apply_zscore(df: DataFrame) -> DataFrame:
    """Apply training-fitted z-score standardization."""
    out = df
    for col_name in NUM_COLS:
        mean = stats[f"{col_name}_mean"]
        out = out.withColumn(
            f"{col_name}_z",
            (F.col(col_name) - F.lit(mean)) / F.lit(positive_stddev(col_name)),
        )
    return out


train_z = apply_zscore(train_te)
val_z = apply_zscore(val_te)
test_z = apply_zscore(test_te)


### 3.6 Feature Assembly

Materialize a compact set of numeric model features so all candidate models use the same ordered inputs.


In [ ]:
# Assemble model-ready feature columns in a stable order.
FEATURE_EXPRESSIONS = [
    ("bias", F.lit(1.0)),
    ("dist", F.col("trip_distance_km_z")),
    ("dur", F.col("duration_min_z")),
    ("pc", F.col("passenger_count_z")),
    ("hr", F.col("hour_z")),
    ("mo", F.col("month_z")),
    ("dw", F.col("dow_z")),
    ("color_te", F.col("color_te_z")),
    ("pu_te", F.col("pu_te_z")),
    ("do_te", F.col("do_te_z")),
]
FEATURE_COLS = [name for name, _ in FEATURE_EXPRESSIONS]
FEAT_COLS = FEATURE_COLS


def with_features(df: DataFrame) -> DataFrame:
    """Materialize compact numeric feature columns for model training."""
    out = df
    for name, expr in FEATURE_EXPRESSIONS:
        out = out.withColumn(name, expr.cast("double"))
    return out.select(*FEATURE_COLS, "total_amount", "label_robust")


train_f = with_features(train_z)
val_f = with_features(val_z)
test_f = with_features(test_z)

print("Feature dimension:", len(FEATURE_COLS))


### 3.7 Candidate Models

Train lightweight models with the same feature set and compare RMSE on both **true fare labels** and **robust capped labels**. Optional diagnostics are controlled by `RUN_MODEL_DIAGNOSTICS`.


In [ ]:
# Model A: closed-form ridge regression.
lam = 1e-3
p = len(FEAT_COLS)

# Aggregate the normal-equation terms against the robust label.
agg_exprs = []
for i, ci in enumerate(FEAT_COLS):
    for j, cj in enumerate(FEAT_COLS):
        if j >= i:
            agg_exprs.append(F.sum(F.col(ci) * F.col(cj)).alias(f"G_{i}_{j}"))

b_exprs = [
    F.sum(F.col(ci) * F.col("label_robust")).alias(f"b_{i}")
    for i, ci in enumerate(FEAT_COLS)
]
row = train_f.agg(*(agg_exprs + b_exprs)).collect()[0].asDict()

G = np.zeros((p, p), dtype=float)
for i in range(p):
    for j in range(i, p):
        G[i, j] = float(row[f"G_{i}_{j}"])
        G[j, i] = G[i, j]

b = np.array([float(row[f"b_{i}"]) for i in range(p)], dtype=float)
w_A = np.linalg.solve(G + lam * np.eye(p), b)

val_A = predict_linear(val_f, FEAT_COLS, w_A, output_col="prediction_A")
test_A = predict_linear(test_f, FEAT_COLS, w_A, output_col="prediction_A")

rmse_val_A_true = rmse(val_A, "prediction_A", "total_amount")
rmse_test_A_true = rmse(test_A, "prediction_A", "total_amount")
rmse_val_A_robust = rmse(val_A, "prediction_A", "label_robust")
rmse_test_A_robust = rmse(test_A, "prediction_A", "label_robust")

print(
    f"Model A RMSE (TRUE)   - val: {rmse_val_A_true:.3f} | "
    f"test: {rmse_test_A_true:.3f}"
)
print(
    f"Model A RMSE (ROBUST) - val: {rmse_val_A_robust:.3f} | "
    f"test: {rmse_test_A_robust:.3f}"
)


In [ ]:
# Model B: ridge regression with gradient descent.
gd_lr = 0.1
gd_epochs = 5
lam_gd = 1e-3
gd_sample_frac = 0.05

train_gd = train_f.select(*FEAT_COLS, F.col("total_amount").alias("label"))
if gd_sample_frac < 1.0:
    train_gd = train_gd.sample(False, gd_sample_frac, seed=42)

w_B = np.zeros(len(FEAT_COLS), dtype=float)

for epoch in range(gd_epochs):
    pred_col = None
    for i, col_name in enumerate(FEAT_COLS):
        term = F.lit(float(w_B[i])) * F.col(col_name)
        pred_col = term if pred_col is None else pred_col + term

    gd_with_pred = (
        train_gd.withColumn("pred", pred_col)
        .withColumn("label_capped", F.least(F.col("label"), F.lit(CAP)))
        .withColumn("resid", F.col("pred") - F.col("label_capped"))
    )

    grad_exprs = [
        F.sum(F.col(col_name) * F.col("resid")).alias(f"g_{i}")
        for i, col_name in enumerate(FEAT_COLS)
    ]
    row = gd_with_pred.agg(*grad_exprs, F.count(F.lit(1)).alias("n")).collect()[0]

    n_rows = float(row["n"]) if row["n"] else 1.0
    gradient = np.array(
        [float(row[f"g_{i}"]) for i in range(len(FEAT_COLS))],
        dtype=float,
    )
    gradient *= 2.0 / n_rows

    regularization = lam_gd * w_B
    regularization[0] = 0.0
    gradient += 2.0 * regularization

    w_B = w_B - gd_lr * gradient
    print(f"Epoch {epoch + 1}/{gd_epochs}: |w|={np.linalg.norm(w_B):.3f}")

val_B = predict_linear(val_f, FEAT_COLS, w_B, output_col="prediction_B")
test_B = predict_linear(test_f, FEAT_COLS, w_B, output_col="prediction_B")

rmse_val_B_true = rmse(val_B, "prediction_B", "total_amount")
rmse_test_B_true = rmse(test_B, "prediction_B", "total_amount")
rmse_val_B_robust = rmse(val_B, "prediction_B", "label_robust")
rmse_test_B_robust = rmse(test_B, "prediction_B", "label_robust")

print(
    f"Model B RMSE (TRUE)   - val: {rmse_val_B_true:.3f} | "
    f"test: {rmse_test_B_true:.3f}"
)
print(
    f"Model B RMSE (ROBUST) - val: {rmse_val_B_robust:.3f} | "
    f"test: {rmse_test_B_robust:.3f}"
)


In [ ]:
# Model C: robust Huber regression.
delta = 20.0
lam = 1e-3
lr = 0.05
epochs = 10
sample_frac = 0.1

train_h = train_f.select(*FEAT_COLS, F.col("total_amount").alias("label"))
if sample_frac < 1.0:
    train_h = train_h.sample(False, sample_frac, seed=42)

w_C = np.zeros(len(FEAT_COLS), dtype=float)

for epoch in range(epochs):
    pred = None
    for i, col_name in enumerate(FEAT_COLS):
        term = F.lit(float(w_C[i])) * F.col(col_name)
        pred = term if pred is None else pred + term

    residual = pred - F.col("label")
    psi = F.when(F.abs(residual) <= F.lit(delta), residual).otherwise(
        F.lit(delta) * F.signum(residual)
    )

    train_h_pred = train_h.withColumn("pred", pred).withColumn("psi", psi)

    grad_exprs = [
        F.sum(F.col(col_name) * F.col("psi")).alias(f"g_{i}")
        for i, col_name in enumerate(FEAT_COLS)
    ]
    row = train_h_pred.agg(*grad_exprs, F.count(F.lit(1)).alias("n")).collect()[0]

    n_rows = float(row["n"]) if row["n"] else 1.0
    gradient = np.array(
        [float(row[f"g_{i}"]) for i in range(len(FEAT_COLS))],
        dtype=float,
    )
    gradient /= max(n_rows, 1.0)

    regularization = lam * w_C
    regularization[0] = 0.0
    gradient += regularization

    w_C = w_C - lr * gradient
    print(f"[Huber] Epoch {epoch + 1}/{epochs} |w|={np.linalg.norm(w_C):.3f}")

val_C = predict_linear(val_f, FEAT_COLS, w_C, output_col="prediction_C")
test_C = predict_linear(test_f, FEAT_COLS, w_C, output_col="prediction_C")

rmse_val_C_true = rmse(val_C, "prediction_C", "total_amount")
rmse_test_C_true = rmse(test_C, "prediction_C", "total_amount")
rmse_val_C_robust = rmse(val_C, "prediction_C", "label_robust")
rmse_test_C_robust = rmse(test_C, "prediction_C", "label_robust")

print(
    f"Model C (Huber) RMSE (TRUE)   - val: {rmse_val_C_true:.3f} | "
    f"test: {rmse_test_C_true:.3f}"
)
print(
    f"Model C (Huber) RMSE (ROBUST) - val: {rmse_val_C_robust:.3f} | "
    f"test: {rmse_test_C_robust:.3f}"
)


In [ ]:
# Model D: fallback histogram tree.
BFEATS = ["dist", "dur", "pu_te", "do_te"]
quantiles = [0.2, 0.4, 0.6, 0.8]

quantile_array = ",".join(str(q) for q in quantiles)
exprs = [
    F.expr(
        f"percentile_approx({col_name}, array({quantile_array}), 10000)"
    ).alias(f"q_{col_name}")
    for col_name in BFEATS
]
qs = train_f.agg(*exprs).first().asDict()


def bin_expr(col_name: str, qvals: list[float]):
    """Convert a continuous feature into a five-bin integer expression."""
    return (
        F.when(F.col(col_name) <= F.lit(qvals[0]), F.lit(0))
        .when(F.col(col_name) <= F.lit(qvals[1]), F.lit(1))
        .when(F.col(col_name) <= F.lit(qvals[2]), F.lit(2))
        .when(F.col(col_name) <= F.lit(qvals[3]), F.lit(3))
        .otherwise(F.lit(4))
        .cast("int")
    )


train_bins = train_f
for col_name in BFEATS:
    train_bins = train_bins.withColumn(
        f"{col_name}_bin",
        bin_expr(col_name, qs[f"q_{col_name}"]),
    )

bin_cols = [f"{col_name}_bin" for col_name in BFEATS]
leaves = (
    train_bins.groupBy(*bin_cols)
    .agg(
        F.avg("total_amount").alias("leaf_pred"),
        F.count("*").alias("n_leaf"),
    )
)


def apply_bins(df: DataFrame) -> DataFrame:
    """Apply train-fitted histogram bins to a model DataFrame."""
    out = df
    for col_name in BFEATS:
        out = out.withColumn(f"{col_name}_bin", bin_expr(col_name, qs[f"q_{col_name}"]))
    return out


val_bins = apply_bins(val_f)
test_bins = apply_bins(test_f)

global_mean = float(train_f.agg(F.avg("total_amount").alias("m")).first()["m"])
val_D = val_bins.join(leaves, on=bin_cols, how="left").withColumn(
    "prediction_D",
    F.coalesce(F.col("leaf_pred"), F.lit(global_mean)),
)
test_D = test_bins.join(leaves, on=bin_cols, how="left").withColumn(
    "prediction_D",
    F.coalesce(F.col("leaf_pred"), F.lit(global_mean)),
)

rmse_val_D_true = rmse(val_D, "prediction_D", "total_amount")
rmse_test_D_true = rmse(test_D, "prediction_D", "total_amount")
rmse_val_D_robust = rmse(val_D, "prediction_D", "label_robust")
rmse_test_D_robust = rmse(test_D, "prediction_D", "label_robust")

print(
    f"Model D summary - TRUE:   val {rmse_val_D_true:.3f} | "
    f"test {rmse_test_D_true:.3f}"
)
print(
    f"Model D summary - ROBUST: val {rmse_val_D_robust:.3f} | "
    f"test {rmse_test_D_robust:.3f}"
)


### 3.8 Model Summary and Artifacts

Compare validation/test RMSE across baseline and candidate models, optionally inspect feature diagnostics, and persist the selected closed-form ridge artifacts for **reproducibility**.


In [ ]:
# Compare candidate models against the baseline.
results = [
    ("Baseline", rmse_val_baseline, rmse_test_baseline),
    ("Model_A_ridge_closed", rmse_val_A_true, rmse_test_A_true),
    ("Model_B_ridge_gd", rmse_val_B_true, rmse_test_B_true),
    ("Model_C_huber", rmse_val_C_true, rmse_test_C_true),
    ("Model_D_tree_fallback", rmse_val_D_true, rmse_test_D_true),
]
best = sorted(results, key=lambda row: row[1])[0]

print("=== RMSE summary on TRUE labels (Val | Test) ===")
for name, val_rmse, test_rmse in results:
    print(f"{name:26s} {val_rmse:8.3f} | {test_rmse:8.3f}")

print()
print(f"Best model by validation RMSE: {best[0]}")
print()
print("ROBUST RMSE diagnostics")
print(f"Model A: val {rmse_val_A_robust:.3f} | test {rmse_test_A_robust:.3f}")
print(f"Model D: val {rmse_val_D_robust:.3f} | test {rmse_test_D_robust:.3f}")


In [ ]:
# Inspect feature correlations and Model A coefficients when diagnostics are enabled.
if RUN_MODEL_DIAGNOSTICS:
    Z_FEATS = [
        "trip_distance_km_z",
        "duration_min_z",
        "passenger_count_z",
        "hour_z",
        "month_z",
        "dow_z",
        "color_te_z",
        "pu_te_z",
        "do_te_z",
    ]

    def corr_with_label(
        df: DataFrame,
        features: list[str],
        label_col: str,
    ) -> DataFrame:
        """Compute feature correlations against one label column."""
        rows = []
        for col_name in features:
            value = df.select(F.corr(F.col(col_name), F.col(label_col)).alias("r")).first()["r"]
            rows.append((col_name, float(value) if value is not None else None))
        return spark.createDataFrame(rows, ["feature", f"corr_{label_col}"])

    corr_true = corr_with_label(train_z, Z_FEATS, "total_amount")
    corr_robust = corr_with_label(train_z, Z_FEATS, "label_robust")

    corr_tbl = (
        corr_true.join(corr_robust, on="feature", how="inner")
        .withColumn("corr_total_amount", F.round("corr_total_amount", 2))
        .withColumn("corr_label_robust", F.round("corr_label_robust", 2))
        .orderBy(F.desc(F.abs("corr_total_amount")))
    )
    display(corr_tbl)

    coef_rows = [(feat, float(weight)) for feat, weight in zip(FEAT_COLS, w_A)]
    coef_df = (
        spark.createDataFrame(coef_rows, ["feature", "coef"])
        .filter(F.col("feature") != "bias")
        .withColumn("abs_coef", F.abs("coef"))
        .orderBy(F.desc("abs_coef"))
        .withColumn("coef", F.format_number("coef", 2))
        .withColumn("abs_coef", F.format_number("abs_coef", 2))
    )
    display(coef_df)


In [ ]:
# Persist selected Model A artifacts to the configured model directory.
weights_df = spark.createDataFrame(
    list(zip(FEAT_COLS, [float(value) for value in w_A.tolist()])),
    ["feature", "weight"],
)
weights_df.write.mode(write_mode).format("delta").save(f"{MODEL_DIR}/weights")

stats_rows = []
for key, value in stats.items():
    col_name, kind = key.rsplit("_", 1)
    stats_rows.append((col_name, kind, float(0.0 if value is None else value)))

schema_stats = T.StructType(
    [
        T.StructField("col", T.StringType(), False),
        T.StructField("stat", T.StringType(), False),
        T.StructField("value", T.DoubleType(), False),
    ]
)
spark.createDataFrame(stats_rows, schema_stats).write.mode(write_mode).format(
    "delta"
).save(f"{MODEL_DIR}/zscore_stats")

for col_name in ["color", "pu_borough", "do_borough"]:
    te_df = te_maps[col_name]
    te_df.write.mode(write_mode).format("delta").save(f"{MODEL_DIR}/te_{col_name}")

meta = {
    "created_at": datetime.datetime.now(datetime.UTC).isoformat(),
    "algo": "ridge_closed_form",
    "lambda": lam,
    "feature_order": FEAT_COLS,
    "y_global_for_TE": float(y_global),
    "te_smoothing_m": te_smoothing,
    "robust_cap": float(CAP),
    "standardized": True,
    "notes": "All stats and TE maps are fit on train data only.",
}
dbutils.fs.put(
    f"{MODEL_DIR}/metadata.json",
    json.dumps(meta, indent=2),
    overwrite=OVERWRITE_TABLES,
)

print(f"Saved Model A artifacts under: {MODEL_DIR}")
display(dbutils.fs.ls(MODEL_DIR))
